# AR(p) synthetic series vs TimesFM 2.5 (Parallel)

Run Monte Carlo replications in parallel across CPU processes, then aggregate summary statistics centrally.

## 1. Setup

This notebook reuses `ar_vs_timesfm_worker.py`, which runs each `(rep, p)` task in a process pool and returns per-task metrics.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
load_dotenv(REPO / ".env")

if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from ar_vs_timesfm_worker import run_monte_carlo_parallel

In [2]:
N_REPLICATIONS = 4
AR_ORDERS = (5,)
N = 720
K_FIRST = 360

# Keep this modest: each worker loads its own TimesFM model instance.
MAX_WORKERS = max(4, os.cpu_count() or 4)
print(MAX_WORKERS)

VERBOSE_PROGRESS = True
PROGRESS_EVERY_TASKS = 1

# Print worker lifecycle and periodic per-worker task progress.
LIVE_WORKER_UPDATES = True
WORKER_PROGRESS_EVERY_STEPS = 50

8


## 2. Run Parallel Monte Carlo

In [3]:
results = run_monte_carlo_parallel(
    n_replications=N_REPLICATIONS,
    n=N,
    k_first=K_FIRST,
    repo_root=REPO,
    ar_orders=AR_ORDERS,
    max_workers=MAX_WORKERS,
    verbose_progress=VERBOSE_PROGRESS,
    progress_every_tasks=PROGRESS_EVERY_TASKS,
    live_worker_updates=LIVE_WORKER_UPDATES,
    worker_progress_every_steps=WORKER_PROGRESS_EVERY_STEPS,
)

summary = (
    results.groupby("p_dgp", as_index=False)
    .agg(
        n=("rep", "count"),
        mse_ar_mean=("mse_ar", "mean"),
        mse_ar_std=("mse_ar", "std"),
        mse_timesfm_mean=("mse_timesfm", "mean"),
        mse_timesfm_std=("mse_timesfm", "std"),
        dm_pvalue_median=("dm_pvalue_two_sided", "median"),
        dm_reject_rate_5pct=("dm_pvalue_two_sided", lambda s: float(np.mean(s < 0.05))),
        ar_fit_failures_total=("ar_fit_failures", "sum"),
    )
)

overall = pd.DataFrame(
    {
        "n_tasks": [len(results)],
        "n_replications": [N_REPLICATIONS],
        "max_workers": [MAX_WORKERS],
        "mse_ar_overall": [float(results["mse_ar"].mean())],
        "mse_timesfm_overall": [float(results["mse_timesfm"].mean())],
        "dm_pvalue_median_overall": [float(results["dm_pvalue_two_sided"].median())],
    }
)

display(summary)
display(overall)
display(results.head(8))

TypeError: run_monte_carlo_parallel() got an unexpected keyword argument 'ar_orders'

## 3. Save Outputs

In [ ]:
out_dir = HERE / "output"
out_dir.mkdir(parents=True, exist_ok=True)

path_rep = out_dir / "ar_vs_timesfm_parallel_replications.csv"
path_sum = out_dir / "ar_vs_timesfm_parallel_summary.csv"
path_overall = out_dir / "ar_vs_timesfm_parallel_overall.csv"

results.to_csv(path_rep, index=False)
summary.to_csv(path_sum, index=False)
overall.to_csv(path_overall, index=False)

print(path_rep)
print(path_sum)
print(path_overall)